In [ ]:
import os
import pandas as pd

from dotenv import load_dotenv
import requests as r

In [ ]:
load_dotenv()

api_key = os.getenv("YOUTUBE_API_KEY")

In [ ]:
url=f"https://youtube.googleapis.com/youtube/v3/channels?part=contentDetails&forHandle=freecodecamp&key={api_key}"
reponse = r.get(url)
print(reponse.json())
uploads_playlist_id=reponse.json()["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
print(uploads_playlist_id)

{'kind': 'youtube#channelListResponse', 'etag': 'wtRwIEZiIy1djWDJkJeLcqmnGeE', 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5}, 'items': [{'kind': 'youtube#channel', 'etag': '8EMaiLPLfF0tEEo6tOpVO5oMep8', 'id': 'UC8butISFwT-Wl7EV0hUK0BQ', 'contentDetails': {'relatedPlaylists': {'likes': '', 'uploads': 'UU8butISFwT-Wl7EV0hUK0BQ'}}}]}
UU8butISFwT-Wl7EV0hUK0BQ


In [ ]:
url_statistics=f"https://youtube.googleapis.com/youtube/v3/channels?part=statistics&forHandle=freecodecamp&key={api_key}"
reponde_st=r.get(url_statistics)
print(reponde_st.json())
url_snnipets =f"https://youtube.googleapis.com/youtube/v3/channels?part=snippet&forHandle=freecodecamp&key={api_key}"
reponde_sn=r.get(url_snnipets)
print(reponde_sn.json())



{'kind': 'youtube#channelListResponse', 'etag': '3r5xmRVMX1jqlexRZvUSvFDV5B4', 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5}, 'items': [{'kind': 'youtube#channel', 'etag': '0mCEB37b4vY1s4l90mOrPhyacIE', 'id': 'UC8butISFwT-Wl7EV0hUK0BQ', 'statistics': {'viewCount': '966539333', 'subscriberCount': '11600000', 'hiddenSubscriberCount': False, 'videoCount': '2221'}}]}
{'kind': 'youtube#channelListResponse', 'etag': 'iM7lDXBOAJwDuWtsheaDmOs9zMg', 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5}, 'items': [{'kind': 'youtube#channel', 'etag': '_Nl1c_lx4ZefffzHSON6qPQwqcs', 'id': 'UC8butISFwT-Wl7EV0hUK0BQ', 'snippet': {'title': 'freeCodeCamp.org', 'description': 'Learn math, programming, and computer science for free. A 501(c)(3) tax-exempt charity. We also run a free learning interactive platform at freecodecamp.org', 'customUrl': '@freecodecamp', 'publishedAt': '2014-12-16T21:18:48Z', 'thumbnails': {'default': {'url': 'https://yt3.ggpht.com/ytc/AIdro_lGRc-05M2OoE1ejQdxeFhyP7OkJg9h4

In [ ]:
videos = []
next_page = ""

#pagination
while True:
    url = f"https://youtube.googleapis.com/youtube/v3/playlistItems?part=snippet&playlistId={uploads_playlist_id}&maxResults=50&pageToken={next_page}&key={api_key}"
    
    data = r.get(url).json()

    video_ids = []

    # les infos :
    for item in data["items"]:
        sn = item["snippet"]
        vid = sn["resourceId"]["videoId"]

        videos.append({
            "title": sn["title"],
            "video_id": vid,
            "published_at": sn["publishedAt"]
        })

        video_ids.append(vid)

    # 📊 stats (batch request)
    stats_url = f"https://youtube.googleapis.com/youtube/v3/videos?part=statistics&id={','.join(video_ids)}&key={api_key}"
    stats_data = r.get(stats_url).json()

    stats_dict = {
        item["id"]: item["statistics"]
        for item in stats_data.get("items", [])
    }

    # 🔗 merge stats
    for v in videos[-len(video_ids):]:
        stats = stats_dict.get(v["video_id"], {})

        v["views"] = int(stats.get("viewCount", 0))
        v["likes"] = int(stats.get("likeCount", 0))
        v["comments"] = int(stats.get("commentCount", 0))

    # next page
    next_page = data.get("nextPageToken")

    if not next_page:
        break

#  DataFrame
df = pd.DataFrame(videos)

df.head(100)

,title,video_id,published_at,views,likes,comments
0,"IT Fundamentals Course – Hardware, Cloud, DevO...",4m9j6hlbf4g,2026-04-28T14:38:54Z,2698,497,24
1,Hot take from the fCC podcast: Use LLMs but do...,oHAiHuEVo8A,2026-04-28T12:05:25Z,915,11,0
2,We all know AI can be...divisive. Justin lays ...,38-3cShsQBc,2026-04-27T12:17:33Z,7711,70,1
3,"Ah, the age-old question. Who can relate?",OnLdZrMcgGU,2026-04-26T12:43:12Z,2711,23,0
4,"Test-driven development is helpful, but it mis...",udA06kmorT0,2026-04-25T12:32:23Z,3636,44,3
...,...,...,...,...,...,...
95,Contributing to open source has many benefits ...,4RGkpzTJjfg,2026-02-25T13:29:55Z,4603,36,2
96,Python Essentials for AI Agents – Tutorial,UsfpzxZNsPo,2026-02-25T11:00:52Z,159724,5721,151
97,Closures in JavaScript explained with a simple...,U6-RekkuORI,2026-02-24T13:05:28Z,21573,248,6
98,Learn Notion – Full Course for Beginners,bB5eA7vU9W8,2026-02-24T11:00:57Z,73438,2577,188


In [ ]:
import json
import os

os.makedirs("data", exist_ok=True)

data_json = df.to_dict(orient="records")

with open("data/videos.json", "w", encoding="utf-8") as f:
    json.dump(data_json, f, indent=4, ensure_ascii=False)

In [ ]:
import pandas as pd
# 
df["published_at"] = pd.to_datetime(df["published_at"])

df["views"] = pd.to_numeric(df["views"], errors="coerce")
df["likes"] = pd.to_numeric(df["likes"], errors="coerce")
df["comments"] = pd.to_numeric(df["comments"], errors="coerce")

df = df.fillna(0)

# 
df["views"] = df["views"].astype(int)
df["likes"] = df["likes"].astype(int)
df["comments"] = df["comments"].astype(int)

In [ ]:
df.to_csv("data/videos.csv", index=False)